# Recycling Points in Berlin – Data Extraction & Normalization

This notebook:
- Extracts recycling locations from OpenStreetMap using OSMnx
- Interprets granular `recycling:*` tags into human-readable categories
- Cleans and consolidates noisy OSM metadata
- Produces a user-facing, analysis-ready GeoDataFrame

Design principles:
- Do not invent information
- Prefer structured tags over free text
- Preserve provenance where ambiguity exists


## Imports

In [ ]:
#%pip install libraries
#%pip install osmnx geopandas pandas
#%pip install geopy
#%pip install --upgrade certifi
#%pip install psycopg2-binary sqlalchemy

In [2]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np

from geopy.geocoders import Nominatim
from shapely.geometry import Point, Polygon
from time import sleep

import re

import psycopg2
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings("ignore")

In [3]:
# Give me all places tagged as OpenStreetMap – Points of interest tagged as amenity=recycling.
# tags filter for only features with 

tags = {"amenity": "recycling"}

In [4]:
# ✅ Enables caching: Speeds up repeated queries
# 🖥 Logs details to the console (helpful for debugging)
"""ox.settings.use_cache = True
ox.settings.log_console = True"""

'ox.settings.use_cache = True\nox.settings.log_console = True'

## Data Source Strategy

Initial extraction from OpenStreetMap was performed once and saved to disk.
To ensure reproducibility and avoid repeated API calls, the notebook
continues from the persisted GeoJSON below.

In [ ]:
# Fetch Supermarkets from Berlin from OSM using the tag "shop=supermarket"
"""df = ox.features.features_from_place("Berlin, Germany", tags=tags)"""

In [ ]:
"""df = df.reset_index()
df.head()"""

In [ ]:
"""df.to_file("../sources/raw_recycling_points.geojson", driver="GeoJSON")"""

In [ ]:
"""df.to_csv("../sources/raw_recycling_points.csv", index=False)"""

## Load Persisted OSM Extract

To ensure reproducibility and avoid repeated OpenStreetMap queries,
the analysis continues from a previously saved GeoJSON file.


In [ ]:
# Load persisted OSM extract to avoid repeated network queries
df = gpd.read_file("../sources/raw_recycling_points.geojson")
df.head()

In [ ]:
gdf = df.copy()

In [ ]:
# Display basic info
print(f"Number of recycling points entries fetched: {len(gdf)}")
gdf.head(3)

In [ ]:
for col in gdf.columns:
    print(col)

In [ ]:
empty_cols = []      # columns with no data
non_empty_cols = []  # columns with data

for col in gdf.columns:
    uniques = gdf[col].unique()
    
    if len(uniques) > 0:
        non_empty_cols.append(col)
        print(f"\n==== {col} ====")
        print(uniques)
    else:
        empty_cols.append(col)

print("\n\nColumns with NO data:")
print(empty_cols)

## Understanding OSM Recycling Tags

OpenStreetMap models recycling using a hierarchical tag structure:

- `recycling:<item>` → whether an item is accepted (yes / no / customers)
- `recycling:<item>:<subtype>` → item-specific details (e.g. glass color)

Examples:
- `recycling:glass_bottles = yes`
- `recycling:glass_bottles:colour = white;green`

The logic below separates:
- base materials
- material subtypes
so they can be interpreted consistently.


In [ ]:
recycling_cols = [
    c for c in gdf.columns 
    if c.startswith("recycling:") and c.count(":") == 1
]
recycling_cols_sub = [
    c for c in gdf.columns 
    if c.startswith("recycling:") and c.count(":") == 2
]

In [ ]:
def dedup_preserve_order(lst):
    seen = set()
    ordered = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            ordered.append(item)
    return ordered

In [ ]:
def extract_recycling(row):
    """
    Parse OSM recycling tags into three explicit categories:

    - accepted: materials explicitly accepted at this location (including explicitly listed subtypes such as glass colors)
    - not_accepted: materials explicitly rejected
    - other: ambiguous or unclear acceptance

    Design choices:
    - Preserve order of appearance (OSM tagging is human-entered)
    - Do not collapse subtype information (e.g. glass colors)
    - Avoid inferring acceptance where tags are ambiguous
    """
    accepted = []
    not_accepted = []
    other = []

    # 1. Base items (recycling:glass_bottles)
    for col in recycling_cols:
        base = col.split(":")[1]
        val = row[col]

        if isinstance(val, str):
            if val == "yes" or "yes" in val.split(";"):
                accepted.append(base)
            elif val == "no" or "no" in val.split(";"):
                not_accepted.append(base)
            elif val == "customers":
                accepted.append(val + '_' + base)
            else:
                other.append(val + '_' + base)

    # 2. Sub-items (recycling:glass_bottles:colour)
    for col in recycling_cols_sub:
        _, base, subtype = col.split(":")

        val = row[col]

        if not isinstance(val, str):
            continue

        # Case: color list: "white;green;brown"
        colors = [x.strip() for x in val.split(";")]

        # Case: value is "yes" meaning "all colors allowed" (rare)
        if val == "yes":
            accepted.append(f"{base}_unknown")
            continue
        elif val == "no":
            not_accepted.append(f"{base}_unknown")
            continue

        # Normal case: add each color explicitly
        for color in colors:
            if color:  # avoid empty strings
                accepted.append(f"{base}_{color}")

    # Remove duplicates but preserve order
    accepted = dedup_preserve_order(accepted)
    not_accepted = dedup_preserve_order(not_accepted)
    other = dedup_preserve_order(other)

    return accepted, not_accepted, other

In [ ]:
gdf["accepted_recycling_items"], gdf["not_accepted_recycling_items"], gdf["other_recycling_items"] = zip(*gdf.apply(extract_recycling, axis=1))

In [ ]:
gdf.head()

In [ ]:
# Replace empty lists with NA
new_cols = ["accepted_recycling_items", "not_accepted_recycling_items", "other_recycling_items"]

for col in new_cols:
    gdf[col] = gdf[col].apply(lambda v: pd.NA if isinstance(v, list) and len(v) == 0 else v)

In [ ]:
gdf.head()

In [ ]:
gdf.loc[
    (gdf["recycling:glass_bottles"] == "yes") &
    (gdf["recycling:glass_bottles:colour"].notna()),
    ["name", "accepted_recycling_items", "not_accepted_recycling_items", "other_recycling_items", "recycling:glass_bottles", "recycling:glass_bottles:colour"]]

In [ ]:
gdf_clean = gdf.drop(columns=recycling_cols + recycling_cols_sub)
gdf_clean.head()

In [ ]:
gdf_clean.info()

## Understanding the data better before grouping

In [ ]:
gdf[gdf["recycling:paper"] == "customers"]

In [ ]:
gdf[gdf["recycling:clothes"] == "no"]

In [ ]:
gdf[gdf["recycling:batteries"] == "no"]

In [ ]:
gdf[gdf["not_accepted_recycling_items"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

### Analyze the cleaned dataset

In [ ]:
gdf_clean[gdf_clean["id"]==26867409]

In [ ]:
for col in gdf_clean.columns:
    print(col)

In [ ]:
gdf_clean[["element","id","recycling_type", "accepted_recycling_items", "not_accepted_recycling_items", "other_recycling_items"]].head()

In [ ]:
df.geometry.iloc[0]
type(df.geometry.iloc[0])
df.geometry.geom_type.unique()

In [ ]:
help(ox.features.features_from_place)
help(ox.features)

In [ ]:
df.iloc[0].to_frame()

## Column Grouping Strategy

OSM metadata is highly heterogeneous.
To clean systematically, columns are grouped by semantic purpose
(e.g. identity, address, contact, accessibility).

Each group is:
- analyzed independently
- normalized using group-specific rules
- collapsed into user-facing fields


In [ ]:
# Define groups of columns for analysis

groups = {
    1: ["element", "id"],
    2: ["geometry"],
    3: ["name", "operator", "operator:short", "short_name", "owner", "alt_name",
        "name:de", "operator:type", "man_made"],
    4: ["addr:street", "addr:housenumber", "addr:postcode", "addr:suburb",
        "description", "ref", "position"],
    5: ["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email"],
    6: ["website", "contact:website", "operator:website"],
    7: ["not_accepted_recycling_items", "accepted_recycling_items", "recycling_type",
        "material", "colour", "green", "waste"],
    8: ["access", "wheelchair", "obstacle:parking", "lit", "addr:floor", "level",
        "indoor", "covered", "count", "barrier"],
    9: ["opening_hours", "collection_times", "access:conditional"],
    10: ["source", "note", "fixme", "status", "landuse"],
}

In [ ]:
# Filter df to include only columns that belong to at least one group
all_group_cols = [col for cols in groups.values() for col in cols]

In [ ]:
# Select only relevant columns
df_selected_clean = gdf_clean[ [col for col in gdf_clean.columns if col in all_group_cols] ]
df_selected_clean.head()

In [ ]:
def analyze_group(df, cols):
    """
    Exploratory analysis helper.

    Purpose:
    - Inspect sparsity and overlap in related columns
    - Identify conflicting or redundant tags
    - Inform cleaning decisions downstream

    Not used in final dataset generation.
    """
    subset = df[cols]
    summary = {}

    # --- Missing values per column ---
    summary["missing_per_column"] = subset.isna().sum().to_dict()

    # --- Unique values per column ---
    summary["unique_values_per_column"] = {
        c: subset[c].dropna().unique().tolist() for c in cols
    }

    # --- Identify rows with >1 filled value ---
    # Define what counts as "filled"
    def is_filled(v):
        if pd.isna(v):
            return False
        if isinstance(v, list) and len(v) == 0:
            return False
        if isinstance(v, str) and v.strip() == "":
            return False
        return True

    filled_mask = subset.applymap(is_filled)
    filled_count = filled_mask.sum(axis=1)

    # Correct: rows where more than one column has a filled value
    summary["rows_with_multiple_filled"] = subset[filled_count > 1]

    return summary

### Group 2: Position

In [ ]:
position_cols = ["geometry"]

In [ ]:
df_selected_clean[df_selected_clean["geometry"].notna()][
 ["id"] + position_cols]

In [ ]:
# Ensure the GeoDataFrame uses WGS84 (longitude, latitude)
# This is required so x = longitude and y = latitude are correct
df_selected_clean = df_selected_clean.set_crs(epsg=4326, allow_override=True)

In [ ]:
# Create empty latitude and longitude columns
# These will be filled for both POINT and POLYGON geometries
df_selected_clean["longitude"] = None
df_selected_clean["latitude"] = None

In [ ]:
# Handle POINT geometries
# For POINT objects, longitude and latitude can be read directly
point_mask = df_selected_clean.geometry.type == "Point"

df_selected_clean.loc[point_mask, "longitude"] = df_selected_clean.loc[point_mask, "geometry"].x
df_selected_clean.loc[point_mask, "latitude"] = df_selected_clean.loc[point_mask, "geometry"].y

In [ ]:
# Handle POLYGON geometries
# For POLYGON objects, compute the centroid (geometric center)
# This represents the average spatial location of the polygon
polygon_mask = df_selected_clean.geometry.type == "Polygon"

# Calculate centroids for polygon geometries
df_selected_clean.loc[polygon_mask, "centroid"] = (
    df_selected_clean.loc[polygon_mask, "geometry"].centroid
)

# Extract longitude and latitude from the centroid points
df_selected_clean.loc[polygon_mask, "longitude"] = (
    df_selected_clean.loc[polygon_mask, "centroid"].x
)
df_selected_clean.loc[polygon_mask, "latitude"] = (
    df_selected_clean.loc[polygon_mask, "centroid"].y
)

In [ ]:
# Clean up temporary centroid column
df_selected_clean = df_selected_clean.drop(columns=["centroid"])

In [ ]:
df_selected_clean.head()

In [ ]:
df_selected_clean[df_selected_clean["longitude"].isna()][["id", "latitude", "longitude"]]

### Group 3: Operator Name

#### Exploratory inspection (not part of final pipeline)


In [ ]:
group3_report = analyze_group(df_selected_clean, groups[3])
group3_report

In [ ]:
name_cols = ["name", "operator", "operator:short", "short_name", "owner", "alt_name",
        "name:de", "operator:type", "man_made"]

In [ ]:
empty_cols = []      # columns with no data
non_empty_cols = []  # columns with data

for col in name_cols:
    uniques = df_selected_clean[col].unique()
    
    if len(uniques) > 0:
        non_empty_cols.append(col)
        print(f"\n==== {col} ====")
        print(uniques)
    else:
        empty_cols.append(col)

print("\n\nColumns with NO data:")
print(empty_cols)

In [ ]:
df_selected_clean[df_selected_clean["man_made"].notna()][
 ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["operator:type"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["id"]==11015694305	]

In [ ]:
df_selected_clean[df_selected_clean["name:de"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["alt_name"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["owner"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["short_name"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["operator:short"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["operator"].notna()][
     ["id"] + name_cols]

In [ ]:
df_selected_clean[df_selected_clean["name"].notna()][
     ["id"] + name_cols]

#### Apply normalization (final pipeline)

In [ ]:
def combine_names(row):
    """
    Combines name-related columns into:
    - display_name: a single best human-readable label (or pd.NA if none exists)
    - name_metadata: list of all name/operator-related values with provenance
    - entity_type: list of classification-related values

    Design choice:
    - We do NOT invent names from entity types (e.g. man_made).
    - Operator is only used as a fallback identifier when no name-like field exists.
    """

    # --- DISPLAY NAME ---
    display_name = pd.NA

    name_priority = [
        "name",
        "short_name",
        "name:de",
        "alt_name",
    ]

    for c in name_priority:
        val = row.get(c)
        if pd.notna(val):
            display_name = val
            break

    # Use operator only if no name exists
    if pd.isna(display_name):
        for c in ["operator", "operator:short"]:
            val = row.get(c)
            if pd.notna(val):
                display_name = val
                break

    # --- NAME METADATA ---
    metadata_cols = [
        "name",
        "short_name",
        "name:de",
        "alt_name",
        "operator",
        "operator:short",
        "owner",
    ]

    name_metadata = []
    for c in metadata_cols:
        val = row.get(c)
        if pd.notna(val):
            name_metadata.append(f"{c}: {val}")

    if not name_metadata:
        name_metadata = pd.NA

    # --- ENTITY TYPE ---
    type_cols = ["operator:type", "man_made"]

    entity_type = []
    for c in type_cols:
        val = row.get(c)
        if pd.notna(val):
            entity_type.append(f"{c}: {val}")

    if not entity_type:
        entity_type = pd.NA

    return pd.Series(
        {
            "display_name": display_name,
            "name_metadata": name_metadata,
            "entity_type": entity_type,
        }
    )

In [ ]:
# Apply to dataframe
df_selected_clean = df_selected_clean.join(df_selected_clean.apply(combine_names, axis=1))
df_selected_clean.head()

In [ ]:
df_selected_clean[
     ["display_name", "name_metadata", "entity_type"] + name_cols]

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=name_cols, inplace=True)
df_selected_clean.head()

### Group 4: Address

#### Exploratory inspection (not part of final pipeline)

In [ ]:
group4_report = analyze_group(df_selected_clean, groups[4])
group4_report

In [ ]:
address_cols=["addr:street", "addr:housenumber", "addr:postcode", "addr:suburb",
        "description", "ref", "position"]

In [ ]:
df_selected_clean[df_selected_clean["position"].notna()][
 ["id"] + address_cols]

In [ ]:
df_selected_clean[df_selected_clean["ref"].notna()][
 ["id"] + address_cols]

In [ ]:
df_selected_clean[df_selected_clean["description"].notna()][
 ["id"] + address_cols]

In [ ]:
df_selected_clean[df_selected_clean["addr:suburb"].notna()][
 ["id"] + address_cols]

In [ ]:
df_selected_clean[df_selected_clean["addr:postcode"].notna()][
 ["id"] + address_cols]

In [ ]:
df_selected_clean[df_selected_clean["addr:housenumber"].notna()][
 ["id"] + address_cols]

In [ ]:
df_selected_clean[df_selected_clean["addr:street"].notna()][
 ["id"] + address_cols]

#### Apply normalization (final pipeline)

In [ ]:
def build_structured_address(row):
    """
    Build a human-readable street address from structured OSM addr:* tags.

    Rationale:
    - addr:street, addr:housenumber, addr:postcode, addr:suburb
      are the highest-quality, machine-curated fields.
    - We only use these fields here to avoid contaminating
      clean addresses with free-text noise.
    """

    parts = []

    # --- Street + house number ---
    # Street is mandatory for a valid structured address.
    # House number is optional and appended only if present.
    if pd.notna(row["addr:street"]):
        street_part = row["addr:street"]

        if pd.notna(row["addr:housenumber"]):
            street_part += f" {row['addr:housenumber']}"

        parts.append(street_part)

    # --- Postcode + suburb ---
    # These are grouped together as a "city part".
    # Either one may exist independently.
    city_part = []

    if pd.notna(row["addr:postcode"]):
        city_part.append(str(row["addr:postcode"]))

    if pd.notna(row["addr:suburb"]):
        city_part.append(row["addr:suburb"])

    if city_part:
        parts.append(" ".join(city_part))

    # If no structured components were available, return NA
    # so that fallback logic can take over explicitly.
    return ", ".join(parts) if parts else pd.NA

In [ ]:
# Apply row-wise because we are combining multiple columns conditionally
df_selected_clean["full_address"] = df_selected_clean.apply(build_structured_address, axis=1)
df_selected_clean.head()

In [ ]:
df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + address_cols + ["full_address"]]

Address Fallback Logic

Not all OSM entries contain structured `addr:*` tags.
When structured addresses are missing, we fall back to free-text fields
in a controlled priority order, explicitly trading precision for coverage.

In [ ]:
# Backup sources ordered by decreasing reliability
# ref        → often contains a usable street + number
# description→ sometimes contains address-like text
# position   is not used due to low quality, intstead reverse search is applied
fallback_cols = ["ref", "description"]

Filter fallback values with simple heuristics.
We only want strings that look like addresses, e.g.:
- contain a street suffix (str, straße, platz, damm, etc.)
- contain a number
- or contain keywords like Ecke, ggü.

In [ ]:
ADDRESS_PATTERN = re.compile(
    r"""
    (
        # Street names
        \b(str\.|straße|strasse|platz|allee|damm|weg|ufer|ring)\b
        |
        # Intersection / relative location
        \b(ecke|ggü\.?|gegenüber)\b
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

def looks_like_address(value):
    if pd.isna(value):
        return False
    value = str(value).strip()
    if len(value) < 5:
        return False
    return bool(ADDRESS_PATTERN.search(value))

In [ ]:
# Apply fallback ONLY if it passes validation
fallback_series = (
    df_selected_clean[["ref", "description"]]
    .apply(
        lambda row: next(
            (
                val for val in row
                if looks_like_address(val)
            ),
            pd.NA
        ),
        axis=1
    )
    .astype("string")
)

df_selected_clean["full_address"] = (
    df_selected_clean["full_address"]
    .astype("string")
    .fillna(fallback_series)
)

In [ ]:
print(df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + address_cols + ["full_address"]])

In [ ]:
print(df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + ["full_address"]])

##### Nominatim

Reverse search position for address

In [ ]:
# Initialize a new Nominatim geocoder instance
geolocator = Nominatim(user_agent="berlin_recycling_locator",
    timeout=10)

In [ ]:
def reverse_geocode_address(lat, lon):
    try:
        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True,
            language="de"
        )
        sleep(1)

        if not location:
            return pd.NA

        raw = location.raw
        addr = raw.get("address", {})

        parts = []

        # Street + house number
        if addr.get("road"):
            street = addr["road"]
            if addr.get("house_number"):
                street += f" {addr['house_number']}"
            parts.append(street)

        # Postcode + city
        city_part = []
        if addr.get("postcode"):
            city_part.append(addr["postcode"])
        if addr.get("city") or addr.get("town"):
            city_part.append(addr.get("city") or addr.get("town"))

        if city_part:
            parts.append(" ".join(city_part))

        # ✅ Structured address available
        if parts:
            return ", ".join(parts)

        # 🔑 Fallback: human-readable name
        if raw.get("display_name"):
            return raw["display_name"]

        return pd.NA

    except Exception as e:
        print("ERROR:", e)
        return pd.NA

In [ ]:
df_selected_clean["full_address"].describe()

In [ ]:
# Apply only where full_address is missing

missing_address_mask = (
    df_selected_clean["full_address"].isna() &
    df_selected_clean["latitude"].notna() &
    df_selected_clean["longitude"].notna()
)

In [ ]:
missing_address_mask.info()

In [ ]:
df_selected_clean.loc[missing_address_mask, "full_address"] = (
    df_selected_clean.loc[missing_address_mask]
    .apply(
        lambda row: reverse_geocode_address(row["latitude"], row["longitude"]),
        axis=1
    )
)

In [ ]:
df_selected_clean[df_selected_clean["full_address"].isna()]

In [ ]:
print(
    "Addresses filled via Nominatim:",
    missing_address_mask.sum()
)

In [ ]:
df_selected_clean["full_address"].isna().sum()


In [ ]:
df_selected_clean[df_selected_clean["full_address"].notna()][
 ["id"] + address_cols + ["full_address"]]

In [ ]:
df_selected_clean["full_address"].info()

In [ ]:
df_selected_clean["full_address"].describe()

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=address_cols, inplace=True)
df_selected_clean.head()

### Group 5: Contact

#### Exploratory inspection (not part of final pipeline)

In [ ]:
group5_report = analyze_group(df_selected_clean, groups[5])
group5_report

In [ ]:
contact_cols=["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email"] 

In [ ]:
df_selected_clean[df_selected_clean["operator:phone"].notna()][
 ["id"] + contact_cols]

In [ ]:
df_selected_clean[df_selected_clean["contact:phone"].notna()][
 ["id"] + contact_cols]

In [ ]:
df_selected_clean[df_selected_clean["phone"].notna()][
 ["id"] + contact_cols]

In [ ]:
df_selected_clean[df_selected_clean["contact:mobile"].notna()][
 ["id"] + contact_cols]

In [ ]:
df_selected_clean[df_selected_clean["contact:fax"].notna()][
 ["id"] + contact_cols]

In [ ]:
df_selected_clean[df_selected_clean["contact:email"].notna()][
 ["id"] + contact_cols]

#### Apply normalization (final pipeline)

In [ ]:
# Define function to combine specified columns with optional prefix
def combine_columns(row, cols, prefix=True, sep="; "):
    values = []

    for c in cols:
        val = row.get(c)
        if pd.notna(val):
            values.append(f"{c}: {val}" if prefix else str(val))

    return sep.join(values) if values else pd.NA

In [ ]:
# Assign combined phones to new column
phone_cols = [
    "operator:phone",
    "contact:phone",
    "phone",
    "contact:mobile",
    "contact:fax", 
    "contact:email"
]
df_selected_clean["contact_combined"] = df_selected_clean.apply(
    combine_columns,
    axis=1,
    cols=phone_cols
)
df_selected_clean.head()

In [ ]:
# View rows of group 5 where fax is present as a test case
df_selected_clean[df_selected_clean["contact:fax"].notna()][
    ["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email", "contact_combined"]
]

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=["operator:phone", "contact:phone", "phone", "contact:mobile", "contact:fax", "contact:email"], inplace=True)
df_selected_clean.head()

### Group 6: Website

#### Exploratory inspection (not part of final pipeline)

In [ ]:
group6_report = analyze_group(df_selected_clean, groups[6])
group6_report

In [ ]:
website_cols = [
    "website", "contact:website", "operator:website"
]

In [ ]:
df_selected_clean[df_selected_clean["website"].notna()][
 ["id"] + website_cols]

In [ ]:
df_selected_clean[df_selected_clean["contact:website"].notna()][
 ["id"] + website_cols]

In [ ]:
df_selected_clean[df_selected_clean["operator:website"].notna()][
 ["id"] + website_cols]

#### Apply normalization (final pipeline)

In [ ]:
def normalize_website(url):
    """
    Normalize website URLs without inventing information.

    Rules:
    - Preserve existing scheme if present
    - Assume https:// only when scheme is missing
    - Leave NA values untouched
    """
    if pd.isna(url):
        return pd.NA

    url = url.strip()

    if url.startswith(("http://", "https://")):
        return url

    # OSM commonly omits scheme; https is the modern safe default
    return f"https://{url}"

In [ ]:
for col in website_cols:
    df_selected_clean[col] = (
        df_selected_clean[col]
        .astype("string")
        .apply(normalize_website)
    )
df_selected_clean[df_selected_clean["contact:website"].notna()][
 ["id"] + website_cols]

In [ ]:
df_selected_clean["website_combined"] = df_selected_clean.apply(
    combine_columns,
    axis=1,
    cols=website_cols
)
df_selected_clean.head()

In [ ]:
df_selected_clean[df_selected_clean["contact:website"].notna()][
 ["id"] + website_cols + ["website_combined"]]

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=website_cols, inplace=True)
df_selected_clean.head()

### Group 7: Recycling Types

#### Exploratory inspection (not part of final pipeline)

In [ ]:
recycling_type_cols = groups[7]
recycling_type_cols

In [ ]:
df_selected_clean[df_selected_clean["not_accepted_recycling_items"].notna()][
 ["id"] + recycling_type_cols]

In [ ]:
df_selected_clean[df_selected_clean["accepted_recycling_items"].notna()][
 ["id"] + recycling_type_cols]

In [ ]:
df_selected_clean[df_selected_clean["recycling_type"].notna()][
 ["id"] + recycling_type_cols]

In [ ]:
df_selected_clean[df_selected_clean["material"].notna()][
 ["id"] + recycling_type_cols]

In [ ]:
df_selected_clean[df_selected_clean["colour"].notna()][
 ["id"] + recycling_type_cols]

In [ ]:
df_selected_clean[df_selected_clean["id"].isin([764609993	,823788768	,2517677621, 5584924473, 5584924475, 6183509465,8688604786, 11412122516	])]

In [ ]:
df_selected_clean[df_selected_clean["green"].notna()][
 ["id"] + recycling_type_cols]

In [ ]:
df_selected_clean[df_selected_clean["waste"].notna()][
 ["id"] + recycling_type_cols]

#### Apply normalization (final pipeline)

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=["material", "colour", "green", "waste", "recycling_type"], inplace=True)


In [ ]:
df_selected_clean.head

### Group 8: Accessability

#### Exploratory inspection (not part of final pipeline)

In [ ]:
group8_report = analyze_group(df_selected_clean, groups[8])
group8_report

In [ ]:
access_cols = ["access", "wheelchair", "obstacle:parking", "lit", "addr:floor", "level",
        "indoor", "covered", "count", "barrier"]

In [ ]:
df_selected_clean[df_selected_clean["access"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["wheelchair"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["obstacle:parking"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["lit"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["addr:floor"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["level"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["indoor"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["covered"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["count"].notna()][
 ["id"] + access_cols]

In [ ]:
df_selected_clean[df_selected_clean["barrier"].notna()][
 ["id"] + access_cols]

#### Apply normalization (final pipeline)

In [ ]:
# access describes legal or social access restrictions
# We preserve it verbatim because values like "residents" or "permit"
# are meaningful and should not be collapsed.
df_selected_clean["access_restriction"] = (
    df_selected_clean["access"]
    .astype("string")
)

In [ ]:
# Wheelchair access is a first-class accessibility feature in OSM.
# Missing values are treated as NA not "no".
df_selected_clean["wheelchair_access"] = (
    df_selected_clean["wheelchair"]
    .astype("string")
    .fillna(pd.NA)
)

In [ ]:
def build_physical_obstacles(row):
    """
    Consolidate all physical obstruction signals into a single field.

    This reduces column count while preserving explicit obstacle information.
    """
    obstacles = []

    if row["obstacle:parking"] == "yes":
        obstacles.append("parking obstruction")

    if pd.notna(row["barrier"]):
        obstacles.append(row["barrier"])

    return ", ".join(obstacles) if obstacles else pd.NA

In [ ]:
df_selected_clean["physical_obstacles"] = (
    df_selected_clean.apply(build_physical_obstacles, axis=1)
    .astype("string")
)

In [ ]:
def build_environmental_features(row):
    """
    Consolidate environmental accessibility modifiers.

    These features affect usability but are not access restrictions.
    """
    features = []

    if row["lit"] == "yes":
        features.append("lit")

    if row["indoor"] == "yes":
        features.append("indoor")
    elif row["covered"] == "yes":
        features.append("covered")

    return ", ".join(features) if features else pd.NA

In [ ]:
df_selected_clean["environmental_features"] = (
    df_selected_clean.apply(build_environmental_features, axis=1)
    .astype("string")
)


In [ ]:
# addr:floor and level represent the same concept at different granularities.
# We prefer addr:floor when available.
df_selected_clean["floor_level"] = (
    df_selected_clean[["addr:floor", "level"]]
    .bfill(axis=1)
    .iloc[:, 0]
    .astype("Float64")
)

In [ ]:
# Capacity is relevant for accessibility and congestion.
df_selected_clean["unit_count"] = (
    df_selected_clean["count"]
    .astype("Int64")
)

In [ ]:
df_selected_clean.head()

In [ ]:
def build_accessibility_features(row):
    """
    Create a concise, human-readable accessibility summary.
    Derived only from explicit consolidated fields.
    """
    parts = []
    # --- Wheelchair accessibility ---

    if pd.notna(row["wheelchair_access"]):
        if row["wheelchair_access"] == "yes":
            parts.append("Wheelchair accessible")
        elif row["wheelchair_access"] == "limited":
            parts.append("Limited wheelchair access")
        elif row["wheelchair_access"] == "no":
            parts.append("Not wheelchair accessible")

    # --- Environmental features ---
    if pd.notna(row["environmental_features"]):
        parts.append(row["environmental_features"].capitalize())

    # --- Physical obstacles ---
    if pd.notna(row["physical_obstacles"]):
        parts.append(f"Obstacles: {row['physical_obstacles']}")

    # --- Access restrictions ---
    # Only mention restrictions if explicitly present and meaningful
    if pd.notna(row["access_restriction"]) and row["access_restriction"] != "yes":
        parts.append(f"Access restricted ({row['access_restriction']})")

    # Return NA if no features were derived
    return ", ".join(parts) if parts else pd.NA

In [ ]:
df_selected_clean["accessibility_features"] = (
    df_selected_clean.apply(build_accessibility_features, axis=1)
    .astype("string")
)

In [ ]:
df_selected_clean[df_selected_clean["accessibility_features"].notna()]

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=access_cols, inplace=True)
df_selected_clean.head()

### Group 9: Opening Hours

#### Exploratory inspection (not part of final pipeline)

In [ ]:
group9_report = analyze_group(df_selected_clean, groups[9])
group9_report

In [ ]:
opening_times_cols = ["opening_hours", "collection_times", "access:conditional"]

In [ ]:
df_selected_clean[df_selected_clean["opening_hours"].notna()][
 ["id"] + opening_times_cols]

In [ ]:
df_selected_clean[df_selected_clean["collection_times"].notna()][
 ["id"] + opening_times_cols]

In [ ]:
df_selected_clean[df_selected_clean["access:conditional"].notna()][
 ["id"] + opening_times_cols]

#### Apply normalization (final pipeline)

In [ ]:
def build_availability_info(row):
    """
    Consolidates opening hours, collection times, and conditional access
    into a single structured, human-readable string.

    Design principles:
    - No invented data
    - NA-first (returns pd.NA if nothing meaningful exists)
    - Preserves semantic distinctions via labeled segments
    """

    parts = []

    # --- Primary availability ---
    # opening_hours is the canonical OSM tag for general accessibility.
    if pd.notna(row["opening_hours"]):
        parts.append(f"hours={row['opening_hours']}")

    # --- Fallback availability ---
    # collection_times are operational (not user access),
    # but included ONLY if opening_hours is missing to avoid data loss.
    elif pd.notna(row["collection_times"]):
        parts.append(f"hours={row['collection_times']} (collection)")

    # --- Conditional restrictions ---
    # access:conditional modifies availability and must never be merged
    # into hours to avoid misinterpretation.
    if pd.notna(row["access:conditional"]):
        parts.append(f"restrictions={row['access:conditional']}")

    # Return NA if no availability-related information exists
    return " | ".join(parts) if parts else pd.NA

In [ ]:
df_selected_clean["availability_info"] = (
    df_selected_clean
    .apply(build_availability_info, axis=1)
    .astype("string")
)

In [ ]:
df_selected_clean[df_selected_clean["availability_info"].notna()][
 ["id"] + opening_times_cols+ ["availability_info"]]

In [ ]:
"""
Exploratory analysis helper.

Purpose:
- Inspect sparsity and overlap in related columns
- Identify conflicting or redundant tags
- Inform cleaning decisions downstream

Not used in final dataset generation.
"""
df_selected_clean.drop(columns=opening_times_cols, inplace=True)

In [ ]:
df_selected_clean.head(10)

### Group 10: Miscellaneous

#### Exploratory inspection (not part of final pipeline)

In [ ]:
group10_report = analyze_group(df_selected_clean, groups[10])
group10_report

#### Apply normalization (final pipeline)

In [ ]:
# Normalize common source spelling inconsistencies
df_selected_clean["source"] = (
    df_selected_clean["source"]
    .str.lower()
    .str.replace("chageset", "changeset", regex=False)
    .str.strip()
    .astype("string")
)

In [ ]:
# Drop free-text editorial metadata that does not describe the POI itself
df_selected_clean.drop(columns=["note", "fixme"], inplace=True)

In [ ]:
# Map operational status to a clear, NA-safe flag
df_selected_clean["is_operational"] = (
    df_selected_clean["status"]
    .map({
        "functional": True,
        "closed": False
    })
    .astype("boolean")
)

df_selected_clean.drop(columns=["status"], inplace=True)

In [ ]:
df_selected_clean["landuse"] = (
    df_selected_clean["landuse"]
    .astype("string")
)

In [ ]:
df_selected_clean.head()

## Adding district Information

In [ ]:
import os
print(os.getcwd())


In [ ]:
# Load official Berlin districts GeoDataFrame from lor_ortsteile.geojson
berlin_districts_gdf = gpd.read_file("../../mapping/lor_ortsteile.geojson")

In [ ]:
berlin_districts_gdf.head()

In [ ]:

# Create a dedicated geometry for spatial join
df_selected_clean["join_geometry"] = df_selected_clean.geometry

df_selected_clean.loc[polygon_mask, "join_geometry"] = (
    df_selected_clean.loc[polygon_mask, "geometry"].centroid
)

df_selected_clean = df_selected_clean.set_geometry("join_geometry")

In [ ]:
# Spatial join
recycling_point_df_district = gpd.sjoin(
    df_selected_clean,
    berlin_districts_gdf[["BEZIRK", "OTEIL", "spatial_name", "geometry"]],
    how="left",
    predicate="within"
)

In [ ]:
recycling_point_df_district.head()

In [ ]:
##just renaming columns for proper schema
recycling_point_df_district = recycling_point_df_district.rename(columns={
    "BEZIRK": "district",
    "OTEIL": "neighborhood",
    "spatial_name": "neighborhood_id"
}).drop(columns=["index_right"])  # drop district_number if not needed
recycling_point_df_district.head()

In [ ]:
# District mapping (official codes as strings)
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
recycling_point_df_district['district_id'] = recycling_point_df_district['district'].map(district_mapping).astype(str)

# (Optional) Check if some districts were not mapped
unmapped = recycling_point_df_district[~recycling_point_df_district['district'].isin(district_mapping.keys())]['district'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts found:", unmapped)

## Final Dataset Characteristics

The resulting GeoDataFrame contains:
- One row per recycling location
- Human-readable name and address
- Explicit accepted / rejected recycling materials
- Normalized contact and website information
- Geometry preserved for spatial analysis
- District mapping


In [ ]:
recycling_point_df_district.drop(columns=["element", "geometry"], inplace=True) #not needed as part of index in final output and the join_geometry is used asthe geometry

In [ ]:
recycling_point_df_district = recycling_point_df_district.rename(columns={
    "join_geometry": "geometry"
})

In [ ]:
recycling_point_df_district = recycling_point_df_district.set_geometry("geometry")

In [ ]:
recycling_point_df_district = recycling_point_df_district.rename(columns={
    "display_name": "name"
})

In [ ]:
recycling_point_df_district.head()

In [ ]:
recycling_point_df_district["district_id"].isna().sum()

In [ ]:
recycling_point_df_district.geometry.crs

## Exporting processed data to geojson and csv

In [ ]:
recycling_point_df_district.to_file("../sources/final_recycling_points_with_district.geojson", driver="GeoJSON")

In [ ]:
recycling_point_df_district.to_csv("../sources/final_recycling_points_with_district.csv", index=False)

## Populate Database

In [ ]:
# Load persisted OSM extract to avoid repeated network queries
recycling_point_df_district = gpd.read_file("../sources/final_recycling_points_with_district.geojson")
recycling_point_df_district.head()

,id,source,landuse,accepted_recycling_items,not_accepted_recycling_items,longitude,latitude,name,name_metadata,entity_type,...,floor_level,unit_count,accessibility_features,availability_info,is_operational,district,neighborhood,neighborhood_id,district_id,geometry
0,26867409,None,None,[glass_bottles],[<NA>],13.296828,52.5013289,Berlin Recycling,[operator: Berlin Recycling],[<NA>],...,NaN,NaN,None,None,NaN,Charlottenburg-Wilmersdorf,Charlottenburg,0401,11004004,POINT (13.29683 52.50133)
1,254985049,None,None,[clothes],[<NA>],13.3205366,52.4839968,<NA>,[<NA>],[<NA>],...,NaN,NaN,None,None,NaN,Charlottenburg-Wilmersdorf,Wilmersdorf,0402,11004004,POINT (13.32054 52.484)
2,262212828,None,None,[glass_bottles],[<NA>],13.4856806,52.5190209,<NA>,[<NA>],[<NA>],...,NaN,NaN,None,None,NaN,Lichtenberg,Lichtenberg,1103,11011011,POINT (13.48568 52.51902)
3,267093410,None,None,[glass_bottles],[<NA>],13.5928941,52.5084566,<NA>,[<NA>],[<NA>],...,NaN,NaN,None,None,NaN,Marzahn-Hellersdorf,Kaulsdorf,1003,11010010,POINT (13.59289 52.50846)
4,272619379,survey,None,[glass_bottles],[<NA>],13.4556341,52.5218288,<NA>,[<NA>],[<NA>],...,NaN,NaN,None,None,NaN,Friedrichshain-Kreuzberg,Friedrichshain,0201,11002002,POINT (13.45563 52.52183)


In [6]:
export_table = recycling_point_df_district.copy()
export_table.dtypes

id                                 int64
source                            object
landuse                           object
accepted_recycling_items          object
not_accepted_recycling_items      object
longitude                         object
latitude                          object
name                              object
name_metadata                     object
entity_type                       object
full_address                      object
contact_combined                  object
website_combined                  object
access_restriction                object
wheelchair_access                 object
physical_obstacles                object
environmental_features            object
floor_level                      float64
unit_count                       float64
accessibility_features            object
availability_info                 object
is_operational                   float64
district                          object
neighborhood                      object
neighborhood_id 

In [7]:
# 1. ID as VARCHAR-compatible
export_table["id"] = export_table["id"].astype(str)

In [25]:
# 2. Latitude / Longitude casting
export_table["latitude"] = export_table["latitude"].astype(float)
export_table["longitude"] = export_table["longitude"].astype(float)
export_table['is_operational'] = export_table['is_operational'].astype('boolean')


In [ ]:

def normalize_cell(x):
    """
    Convert unsupported types to PostgreSQL-safe values.

    - numpy arrays:
        * empty -> None (NULL)
        * single value -> scalar
        * multiple values -> comma-separated string
    - everything else -> unchanged
    """
    if isinstance(x, np.ndarray):
        if len(x) == 0:
            return None
        if len(x) == 1:
            return None if str(x[0]) in ("<NA>", "NA") else str(x[0])
        return ",".join(map(str, x))
    return x

In [10]:

# Normalize all columns (arrays → strings / NULL)
for col in export_table.columns:
    export_table[col] = export_table[col].apply(normalize_cell)

In [11]:
# Convert geometry (Shapely → WKT string)
export_table["geometry"] = export_table["geometry"].apply(
    lambda g: g.wkt if g is not None else None)

In [12]:
export_table.dtypes

id                               object
source                           object
landuse                          object
accepted_recycling_items         object
not_accepted_recycling_items     object
longitude                       float64
latitude                        float64
name                             object
name_metadata                    object
entity_type                      object
full_address                     object
contact_combined                 object
website_combined                 object
access_restriction               object
wheelchair_access                object
physical_obstacles               object
environmental_features           object
floor_level                     float64
unit_count                      float64
accessibility_features           object
availability_info                object
is_operational                  float64
district                         object
neighborhood                     object
neighborhood_id                  object


In [ ]:
# Input user credentials for database connection
user=''
password=''

In [14]:
# Conection
host = 'localhost'
port = '5433'
database = 'layereddb'
schema='berlin_source_data'

#connection to db after you opened tunnel
engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')

### Usetable tested above with constraints

In [21]:
create_table_query = f"""

DROP TABLE IF EXISTS berlin_source_data.recycling_points;


CREATE TABLE IF NOT EXISTS berlin_source_data.recycling_points (
    -- Core POI schema
    id VARCHAR(20) PRIMARY KEY,
    district_id VARCHAR(20) NOT NULL,

    name VARCHAR(200) NOT NULL DEFAULT 'Unknown',

    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    geometry VARCHAR NOT NULL,

    neighborhood VARCHAR(100) DEFAULT NULL,
    district VARCHAR(100) DEFAULT NULL,
    neighborhood_id VARCHAR(20),

    -- Domain-specific attributes
    entity_type VARCHAR(100) DEFAULT 'Unknown',
    landuse VARCHAR(100) DEFAULT NULL,
    accepted_recycling_items TEXT DEFAULT NULL,
    not_accepted_recycling_items TEXT DEFAULT NULL,
    access_restriction VARCHAR(100) DEFAULT NULL,
    wheelchair_access VARCHAR(100) DEFAULT NULL,
    physical_obstacles TEXT DEFAULT NULL,
    accessibility_features TEXT DEFAULT NULL,

    is_operational BOOLEAN,

    -- Location & availability
    full_address TEXT DEFAULT NULL,
    floor_level INTEGER,
    unit_count INTEGER,
    availability_info TEXT DEFAULT NULL,

    -- Contact & metadata
    contact_combined TEXT DEFAULT NULL,
    website_combined TEXT DEFAULT NULL,
    environmental_features TEXT DEFAULT NULL,
    source VARCHAR(100) DEFAULT 'Unknown',
    name_metadata VARCHAR(200) DEFAULT NULL,

    CONSTRAINT district_id_fk
        FOREIGN KEY (district_id)
        REFERENCES berlin_source_data.districts(district_id)
        ON DELETE RESTRICT
        ON UPDATE CASCADE,

    CONSTRAINT latitude_check
        CHECK (latitude BETWEEN 52.2 AND 52.8),

    CONSTRAINT longitude_check
        CHECK (longitude BETWEEN 12.9 AND 13.9)
);

"""

In [22]:
with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()  # commit the transaction
    print("Table 'recycling_points' created or already exists.")  

Table 'recycling_points' created or already exists.


In [26]:
export_table.to_sql(
    'recycling_points',      
    engine,
    schema=schema,
    if_exists='append', # ✅ keeps table, just inserts data
    index=False
)

print("DataFrame sent to PostgreSQL using .to_sql() with psycopg2!")

DataFrame sent to PostgreSQL using .to_sql() with psycopg2!


In [27]:
query = f"""
SELECT COUNT(*) 
FROM berlin_source_data.recycling_points
WHERE name IS NULL;
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df

,count
0,0


In [28]:
query = f"""
SELECT rp.district_id, di.district_id, COUNT(*) as recycling_points_count
FROM berlin_source_data.recycling_points rp
JOIN berlin_source_data.districts di
ON rp.district_id = di.district_id
GROUP BY rp.district_id, di.district_id
ORDER BY rp.district_id;
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df

,district_id,district_id,recycling_points_count
0,11001001,11001001,241
1,11002002,11002002,185
2,11003003,11003003,320
3,11004004,11004004,165
4,11005005,11005005,107
5,11006006,11006006,193
6,11007007,11007007,133
7,11008008,11008008,250
8,11009009,11009009,340
9,11010010,11010010,363
